rozdělení dat na testovací (20%) a trénovací množinu (80%) (X_train, y_train, X_test, y_test)

In [1]:
import pandas as pd
df = pd.read_csv('/Users/vaclavvyhnalek/Python files/Netflix-churn/netflix_customer_churn.csv')
df.head()

,customer_id,age,gender,subscription_type,watch_hours,last_login_days,region,device,monthly_fee,churned,payment_method,number_of_profiles,avg_watch_time_per_day,favorite_genre
0,a9b75100-82a8-427a-a208-72f24052884a,51,Other,Basic,14.73,29,Africa,TV,8.99,1,Gift Card,1,0.49,Action
1,49a5dfd9-7e69-4022-a6ad-0a1b9767fb5b,47,Other,Standard,0.70,19,Europe,Mobile,13.99,1,Gift Card,5,0.03,Sci-Fi
2,4d71f6ce-fca9-4ff7-8afa-197ac24de14b,27,Female,Standard,16.32,10,Asia,TV,13.99,0,Crypto,2,1.48,Drama
3,d3c72c38-631b-4f9e-8a0e-de103cad1a7d,53,Other,Premium,4.51,12,Oceania,TV,17.99,1,Crypto,2,0.35,Horror
4,4e265c34-103a-4dbb-9553-76c9aa47e946,56,Other,Standard,1.89,13,Africa,Mobile,13.99,1,Crypto,2,0.13,Action


In [2]:
import numpy as np

# Odstranění duplicit
df = df.drop_duplicates()

# Normalizace textových hodnot
text_cols = [
    "gender",
    "subscription_type",
    "region",
    "device",
    "payment_method",
    "favorite_genre"
]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Logicky nesmyslné hodnoty → NaN
df.loc[(df["age"] < 0) | (df["age"] > 120), "age"] = np.nan

df.loc[df["watch_hours"] < 0, "watch_hours"] = np.nan
df.loc[df["avg_watch_time_per_day"] < 0, "avg_watch_time_per_day"] = np.nan

df.loc[df["last_login_days"] < 0, "last_login_days"] = np.nan

df.loc[df["number_of_profiles"] < 0, "number_of_profiles"] = np.nan

df.loc[df["monthly_fee"] < 0, "monthly_fee"] = np.nan

df.isna().sum()

customer_id               0
age                       0
gender                    0
subscription_type         0
watch_hours               0
last_login_days           0
region                    0
device                    0
monthly_fee               0
churned                   0
payment_method            0
number_of_profiles        0
avg_watch_time_per_day    0
favorite_genre            0
dtype: int64

Po provedení kontroly kvality dat nebyly v datasetu identifikovány žádné chybějící hodnoty ani logicky nesmyslné záznamy. Přesto byl preprocessing navržen tak, aby byl robustní i vůči případným chybám v datech.

Rozdělení dat na trénovací a testovací množinu:

In [3]:
from sklearn.model_selection import train_test_split

# features x target
X = df.drop(columns=["churned", "customer_id"])
y = df["churned"]

# train / test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((4000, 12), (1000, 12))

In [4]:
# numerické proměnné
numeric_features = [
    "age",
    "watch_hours",
    "last_login_days",
    "monthly_fee",
    "number_of_profiles",
    "avg_watch_time_per_day"
]

# kategorické proměnné
categorical_features = [
    "gender",
    "subscription_type",
    "region",
    "device",
    "payment_method",
    "favorite_genre"
]

numeric_features, categorical_features

(['age',
  'watch_hours',
  'last_login_days',
  'monthly_fee',
  'number_of_profiles',
  'avg_watch_time_per_day'],
 ['gender',
  'subscription_type',
  'region',
  'device',
  'payment_method',
  'favorite_genre'])

RobustScaler byl zvolen kvůli přítomnosti výrazných outlierů v numerických proměnných (např. watch_hours, avg_watch_time_per_day), které by u StandardScaleru nepřiměřeně ovlivňovaly výpočet průměru a směrodatné odchylky. RobustScaler využívá medián a IQR, a je proto vůči extrémním hodnotám stabilnější.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

# pipeline pro numerické proměnné
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

# pipeline pro kategorické proměnné
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# spojení do jednoho preprocessing kroku
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [6]:
# fit preprocessingu pouze na TRAIN datech
X_train_prepared = preprocessor.fit_transform(X_train)

# aplikace stejného preprocessingu na TEST data
X_test_prepared = preprocessor.transform(X_test)

# kontrola tvarů
X_train_prepared.shape, X_test_prepared.shape

((4000, 35), (1000, 35))

nevyváženost tříd:

In [7]:
# Rozdělení tříd v trénovacích datech
class_counts = y_train.value_counts()
class_ratio = y_train.value_counts(normalize=True)

class_counts, class_ratio

(churned
 1    2012
 0    1988
 Name: count, dtype: int64,
 churned
 1    0.503
 0    0.497
 Name: proportion, dtype: float64)

Dataset je od začátku vyvážený. není proto nutno aplikovat metody pro vyvážení dat.

In [8]:
from joblib import dump

dump(preprocessor, "preprocessor.joblib")

['preprocessor.joblib']